# Comparación: Regresión Logística vs SVM vs KNN
**Dataset:** `Driver_Behavior.csv`



**Consejo:** **NO** escales antes de dividir en train/test. Primero split, luego fit del scaler solo con train.


### Objetivo
Construir y **comparar** tres modelos de clasificación (Logística, KNN y SVM) para predecir el comportamiento/riesgo del conductor.

### Bloques
- Bloque 1: Exploración
- Bloque 2: Preparación
- Bloque 3: Regresión logística
- Bloque 4: KNN
- Bloque 5: SVM
- Bloque 6: Comparación final

### Preguntas iniciales
1. ¿Por qué KNN es sensible al escalado?
2. ¿Qué efecto tiene aumentar el parámetro C en SVM?
3. ¿En qué se diferencia la frontera de decisión de Logística y SVM?
4. ¿Por qué la Accuracy puede no ser una buena métrica en problemas desbalanceados?



## Preparación
1. Coloca el archivo `Driver_Behavior.csv` en la **misma carpeta** que esté tu notebook, o ajusta la ruta.
2. Ejecuta celda a celda.
3. Mantén el notebook limpio: salidas relevantes, gráficos claros, comentarios breves.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

import matplotlib.pyplot as plt


In [2]:
# Carga del dataset
path = "Driver_Behavior.csv"
df = pd.read_csv(path)

display(df.head())
print("Shape:", df.shape)
df.info()


,speed_kmph,accel_x,accel_y,brake_pressure,steering_angle,throttle,lane_deviation,phone_usage,headway_distance,reaction_time,behavior_label
0,36.075011,0.535763,0.708633,23.107812,-3.169956,53.123505,0.851871,1,17.996005,1.400050,Distracted
1,38.090536,0.973764,0.044312,36.961137,-24.380082,36.383904,1.459495,1,29.904182,1.428537,Distracted
2,71.314445,3.638434,0.789375,79.734087,-6.100238,78.110507,0.254723,0,11.126012,0.406950,Aggressive
3,86.485997,2.441366,0.039135,45.007002,17.886191,82.794935,0.911664,0,11.064505,0.539964,Aggressive
4,52.816777,-0.201763,0.560619,38.759612,-4.104323,61.432375,1.591244,1,21.967570,1.369908,Distracted


Shape: (30000, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   speed_kmph        30000 non-null  float64
 1   accel_x           30000 non-null  float64
 2   accel_y           30000 non-null  float64
 3   brake_pressure    30000 non-null  float64
 4   steering_angle    30000 non-null  float64
 5   throttle          30000 non-null  float64
 6   lane_deviation    30000 non-null  float64
 7   phone_usage       30000 non-null  int64  
 8   headway_distance  30000 non-null  float64
 9   reaction_time     30000 non-null  float64
 10  behavior_label    30000 non-null  object 
dtypes: float64(9), int64(1), object(1)
memory usage: 2.5+ MB



# Bloque 1 – Exploración

Completa:

1. Identifica **variable objetivo** (y) y **predictoras** (X).
2. Muestra:
   - `df.describe()` (si tiene sentido)
   - Recuento por clase de la variable objetivo
   - Valores nulos por columna
3. Responde (2–4 líneas cada una):
   - ¿Está balanceada la variable objetivo?
   - ¿Qué variables crees que pueden ser más predictoras?
   - ¿Conviene escalar? ¿Por qué?

💡 Consejo: si la variable objetivo está en texto (p.ej. "Safe"/"Aggressive"), necesitarás codificarla o usarla tal cual según scikit-learn (normalmente admite etiquetas tipo string).


In [9]:
df["behavior_label"] = df["behavior_label"].map({"Distracted": 0, "Aggressive": 1, "Safe": 2})

In [10]:
X = df.drop(["behavior_label"],axis=1)
y = df["behavior_label"]

In [11]:
df.describe()

,speed_kmph,accel_x,accel_y,brake_pressure,steering_angle,throttle,lane_deviation,phone_usage,headway_distance,reaction_time,behavior_label
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.00000
mean,59.986424,1.265818,0.368501,40.767624,-0.040207,55.001223,0.568549,0.333333,23.399177,0.999817,1.00000
std,14.806008,1.026624,0.295654,26.721728,11.384086,21.475323,0.420563,0.471412,11.998469,0.466180,0.81651
min,20.000000,-0.949617,-0.479718,0.003128,-59.989984,20.001444,0.000001,0.000000,5.004359,0.400008,0.00000
25%,49.568893,0.506529,0.116047,18.722464,-6.215165,37.246356,0.234971,0.000000,13.683875,0.625024,0.00000
50%,57.901281,0.831602,0.313145,39.951206,-0.018734,50.066483,0.456616,0.000000,20.133699,0.851295,1.00000
75%,69.242746,1.968167,0.568768,57.914900,6.158074,70.144059,0.810950,1.000000,31.308284,1.396176,2.00000
max,118.439831,5.308924,1.664605,99.994365,53.426806,99.994762,2.425784,1.000000,49.998924,1.999885,2.00000


In [12]:
counts = df["behavior_label"].value_counts()
print(counts)
print(f"\nRatio: {counts.max() / counts.min():.2f}x")

behavior_label
0    10000
1    10000
2    10000
Name: count, dtype: int64

Ratio: 1.00x


In [13]:
df.isnull().sum()

speed_kmph          0
accel_x             0
accel_y             0
brake_pressure      0
steering_angle      0
throttle            0
lane_deviation      0
phone_usage         0
headway_distance    0
reaction_time       0
behavior_label      0
dtype: int64

In [14]:
df["behavior_label"].unique()

array([0, 1, 2])

In [15]:
df.corr(numeric_only=True)["behavior_label"].abs().sort_values(ascending=False)[1:]

phone_usage         0.866025
lane_deviation      0.777857
reaction_time       0.655399
headway_distance    0.593190
brake_pressure      0.419720
accel_y             0.412901
throttle            0.285810
speed_kmph          0.131538
accel_x             0.118126
steering_angle      0.002830
Name: behavior_label, dtype: float64

- ¿Está balanceada la variable objetivo?
   -  La variable objetivo esta balanceada ya que las tres posibilidades de la etiqueta tienen el mismo numero de casos, 1000.

- ¿Qué variables crees que pueden ser más predictoras?
   -  Las variables con mayor correlacion con la columna objetivo como phone_usage, lane_deviation, reaction_time.

- ¿Conviene escalar? ¿Por qué?
   -  Conviene escalar ya que s


# Bloque 2 – Preparación

1. Divide en train/test (80/20), con `random_state=42`.
2. Aplica **StandardScaler**:
   - `fit` SOLO con X_train
   - `transform` en X_train y X_test
3. Justifica por qué el escalado es importante para KNN y SVM.



## Funciones de ayuda

- accuracy
- matriz de confusión
- classification_report
- ROC y AUC (si el modelo ofrece `predict_proba` o `decision_function`)



# Bloque 3 – Regresión Logística

1. Entrena una Regresión Logística.
2. Evalúa con:
   - Accuracy
   - Matriz de confusión
   - Classification report
   - ROC y AUC (si procede)
3. Interpreta:
   - ¿Qué tipo de error es más preocupante en este problema y por qué?



# Bloque 4 – KNN

1. Entrena KNN con `k=5`.
2. Evalúa con las mismas métricas.
3. Prueba al menos 3 valores distintos de k (por ejemplo: 1, 5, 11, 21).
4. Representa **Accuracy vs k** en un gráfico.
5. Elige el k "óptimo" y justifica tu elección.



# Bloque 5 – SVM

1. Entrena:
   - SVM lineal (`kernel="linear"`)
   - SVM RBF (`kernel="rbf"`)
2. Evalúa ambos modelos con las mismas métricas.
3. Modifica el parámetro **C** (por ejemplo: 0.1, 1, 10) y analiza el efecto.
4. Indica qué kernel funciona mejor y por qué.



# Bloque 6 – Comparación final

1. Crea una **tabla comparativa** con:
   - Accuracy
   - Recall (macro o de la clase positiva)
   - F1-score (macro o de la clase positiva)
   - AUC (si es binario)
2. Recomienda un modelo y justifica tu elección.

En problemas de seguridad, suele importar especialmente el **Recall** de la clase de riesgo.



# Preguntas

Responde brevemente:

1. ¿Por qué KNN es sensible al escalado?
2. ¿Qué efecto tiene aumentar el parámetro C en SVM?
3. ¿En qué se diferencia la frontera de decisión de Logística y SVM?
4. ¿Por qué la Accuracy puede no ser una buena métrica en problemas desbalanceados?


# Ayuda para las explicaciones
A la hora de responder a las preguntas, puedes ayudarte de esta pequeña guía.

1. Cada conclusión debe incluir:

    - Una métrica concreta
    - Una comparación explícita
    - Una consecuencia técnica


2. Siempre debes responder

    - ¿Qué significa esa métrica?
    - ¿Por qué importa aquí?
    - ¿Qué implicación tiene?

3. Cuando compares modelos debes responder a 3 preguntas (más o menos):
    - ¿Cuál rinde mejor globalmente? (Accuracy / AUC)
    - ¿Cuál detecta mejor la clase importante? (Recall)
    - ¿Cuál es más estable o generalizable?

>Estas notas son orientativas pero pueden ayudarte a la hora de mejorar tus respuestas.